# Phase 12: Joint Fine-Tuned Weighted ResNet-18 + EfficientNet Attention Fusion
**Diabetic Retinopathy Screening Project**

This notebook implements the final model optimization step:
Allowing the **CNN backbones (top layers of ResNet-18 and EfficientNet-B0)** to fine-tune jointly with the attention fusion head using **differential learning rates**.

### Objectives:
- Unfreeze `resnet.layer4` and `efficientnet.features[6:]`.
- Set backbone learning rate to $10^{-5}$ and fusion head learning rate to $10^{-4}$.
- Train with Class-Weighted Cross-Entropy Loss to counter severe class imbalance.
- Target: Outperform the single-model baseline (Macro F1 > 0.7087, QWK > 0.7628, Acc > 0.7431).

In [ ]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    precision_recall_fscore_support, matthews_corrcoef
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DR_CLASSES = ["No DR", "Mild", "Moderate", "Severe", "PDR"]


In [ ]:
def resolve_first_existing(candidates):
    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None

# Metadata: prefer the CORRECTED split. Every checkpoint we load (weighted
# ResNet-18, EfficientNet-B0, Attention Fusion) was actually trained and
# test-evaluated in notebook 09 against corrected_unified_metadata.csv's
# train/val/test assignment, NOT the plain unified_metadata.csv. Using the
# wrong one risks train/test leakage or an entirely different held-out set.
METADATA_CANDIDATES = [
    "/kaggle/input/datasets/nethrakrishnan/corrected-unified-metadata/corrected_unified_metadata.csv",
    "/kaggle/input/corrected_unified_metadata/corrected_unified_metadata.csv",
    "/kaggle/input/corrected-unified-metadata/corrected_unified_metadata.csv",
    "data/metadata/corrected_unified_metadata.csv",
]
METADATA_PATH = resolve_first_existing(METADATA_CANDIDATES)
if METADATA_PATH is None:
    warnings.warn(
        "corrected_unified_metadata.csv not found on any candidate path — falling back to "
        "unified_metadata.csv. The train/val/test split may NOT match what the checkpoints "
        "were actually trained on."
    )
    METADATA_PATH = resolve_first_existing([
        "/kaggle/input/datasets/nethrakrishnan/unified-metadata/unified_metadata.csv",
        "/kaggle/input/unified-metadata/unified_metadata.csv",
        "/kaggle/input/unified_metadata/unified_metadata.csv",
        "data/metadata/unified_metadata.csv",
    ])
assert METADATA_PATH is not None, "No metadata CSV found among Kaggle/local candidates."
print(f"Using metadata: {METADATA_PATH}")

IMG_ROOT = resolve_first_existing([
    "/kaggle/input/datasets/nethrakrishnan/processed-images/processed/images",
    "/kaggle/input/processed-images/processed/images",
    "/kaggle/input/processed_images/processed/images",
    "/kaggle/input/datasets/aditik1234/processed/processed/images",
    "data/processed/images",
])
assert IMG_ROOT is not None, "No processed-images directory found among Kaggle/local candidates."
print(f"Using image root: {IMG_ROOT}")

# Kaggle writes persist under /kaggle/working; locally we write under models/ + reports/
CKPT_OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "models"
FIG_DIR = "/kaggle/working/figures" if os.path.isdir("/kaggle/working") else "reports/figures/finetuning"
TABLE_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "reports/tables"
os.makedirs(CKPT_OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

df = pd.read_csv(METADATA_PATH, low_memory=False)
model_df = df[df["eligible_for_model"] == True].copy()

train_df = model_df[model_df["model_split"] == "train"].copy().reset_index(drop=True)
val_df = model_df[model_df["model_split"] == "val"].copy().reset_index(drop=True)
test_df = model_df[model_df["model_split"] == "test"].copy().reset_index(drop=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")


In [ ]:
# Class weights computed dynamically from the TRAIN split's inverse class
# frequency (previously hardcoded constants from an unrelated run, which
# silently drift out of sync whenever the split/data changes).
train_labels = train_df["label_dr_standard"].dropna().astype(int).values
class_counts = np.bincount(train_labels, minlength=len(DR_CLASSES)).astype(np.float64)
class_counts = np.clip(class_counts, a_min=1.0, a_max=None)  # avoid div-by-zero on empty classes
inverse_freq = class_counts.sum() / (len(DR_CLASSES) * class_counts)
class_weights = torch.tensor(inverse_freq, dtype=torch.float32).to(device)

print("Train class counts:", class_counts.astype(int).tolist())
print("Class weights:", np.round(inverse_freq, 4).tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)
print("✓ Class-weighted loss configured (weights derived from the current train split)")


In [ ]:
# Model definition with fine-tunable backbones.
# Must mirror notebooks/09_model_building.ipynb's AttentionFusionModel
# EXACTLY (including the 3-layer classifier) so that
# best_weighted_resnet_efficientnet_attention.pth loads with strict=True
# as the starting point for joint fine-tuning.
class JointAttentionFusionModel(nn.Module):
    def __init__(self, resnet_features, efficientnet_features, efficientnet_pool, num_classes=5):
        super().__init__()
        self.resnet_features = resnet_features
        self.efficientnet_features = efficientnet_features
        self.efficientnet_pool = efficientnet_pool

        self.resnet_projection = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256)
        )
        self.efficientnet_projection = nn.Sequential(
            nn.Linear(1280, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256)
        )
        self.attention = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        r_feat = self.resnet_features(x)
        r_feat = torch.flatten(r_feat, 1)
        r_proj = self.resnet_projection(r_feat)

        e_feat = self.efficientnet_features(x)
        e_feat = self.efficientnet_pool(e_feat)
        e_feat = torch.flatten(e_feat, 1)
        e_proj = self.efficientnet_projection(e_feat)

        combined = torch.cat([r_proj, e_proj], dim=1)
        attn_logits = self.attention(combined)
        attn_weights = torch.softmax(attn_logits, dim=1)

        fused = attn_weights[:, 0:1] * r_proj + attn_weights[:, 1:2] * e_proj
        out = self.classifier(fused)
        return out, attn_weights


In [ ]:
# Build fresh backbones, then load pretrained single-branch weights into
# them (these were the backbones the attention-fusion checkpoint itself was
# built on top of in notebook 09).
resnet_raw = models.resnet18(weights=None)
resnet_raw.fc = nn.Linear(resnet_raw.fc.in_features, 5)
effnet_raw = models.efficientnet_b0(weights=None)
effnet_raw.classifier[1] = nn.Linear(effnet_raw.classifier[1].in_features, 5)


def resolve_checkpoint(candidates):
    for p in candidates:
        if p and os.path.exists(p):
            return p
    return None


def load_checkpoint_into(module, path, label, strict=True):
    if path is None:
        warnings.warn(f"[{label}] No checkpoint found — starting from RANDOM initialization.")
        return False
    ckpt = torch.load(path, map_location=device)
    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    module.load_state_dict(state, strict=strict)
    print(f"✓ Loaded {label} checkpoint from: {path}")
    return True


RESNET_CANDIDATES = [
    "/kaggle/input/datasets/nethrakrishnan/dr-checkpoints/best_resnet18_weighted.pth",
    "/kaggle/input/dr-checkpoints/best_resnet18_weighted.pth",
    "/kaggle/input/datasets/aditik1234/new-model-dr/best_resnet18_weighted (1).pth",
    "/kaggle/working/best_resnet18_weighted.pth",
    "models/best_resnet18_weighted.pth",
]
EFFNET_CANDIDATES = [
    "/kaggle/input/datasets/nethrakrishnan/dr-checkpoints/best_efficientnet_b0.pth",
    "/kaggle/input/dr-checkpoints/best_efficientnet_b0.pth",
    "/kaggle/input/datasets/aditik1234/dataset-resent-effifientnet/best_efficientnet_b0.pth",
    "/kaggle/working/best_efficientnet_b0.pth",
    "models/best_efficientnet_b0.pth",
]
ATTN_CANDIDATES = [
    "/kaggle/input/datasets/nethrakrishnan/dr-checkpoints/best_weighted_resnet_efficientnet_attention.pth",
    "/kaggle/input/dr-checkpoints/best_weighted_resnet_efficientnet_attention.pth",
    "/kaggle/input/datasets/aditik1234/new-model-dr/best_weighted_resnet_efficientnet_attention.pth",
    "/kaggle/working/best_weighted_resnet_efficientnet_attention.pth",
    "models/best_weighted_resnet_efficientnet_attention.pth",
]

load_checkpoint_into(resnet_raw, resolve_checkpoint(RESNET_CANDIDATES), "Weighted ResNet-18 backbone")
load_checkpoint_into(effnet_raw, resolve_checkpoint(EFFNET_CANDIDATES), "EfficientNet-B0 backbone")

res_features = nn.Sequential(*list(resnet_raw.children())[:-1])  # [...,layer4, avgpool]; drops only fc
eff_features = effnet_raw.features
eff_pool = nn.AdaptiveAvgPool2d((1, 1))

# Freeze everything, then unfreeze EXACTLY the requested top stages:
# ResNet-18: layer4 only (res_features[-2], since res_features[-1] is avgpool
# and has no learnable parameters).
# EfficientNet-B0: features[6:] (the last 3 of 9 stages -> eff_features[-3:]).
for p in res_features.parameters():
    p.requires_grad = False
for p in res_features[-2].parameters():  # layer4
    p.requires_grad = True

for p in eff_features.parameters():
    p.requires_grad = False
for p in eff_features[-3:].parameters():  # features[6:]
    p.requires_grad = True

joint_model = JointAttentionFusionModel(res_features, eff_features, eff_pool, num_classes=5).to(device)

# Load the ALREADY-TRAINED attention-fusion checkpoint into the joint model
# so fine-tuning continues from it, rather than starting the fusion head
# and projections from scratch.
load_checkpoint_into(joint_model, resolve_checkpoint(ATTN_CANDIDATES), "Attention Fusion (fine-tuning base)")

# Differential parameter groups
backbone_params = [p for p in res_features[-2].parameters() if p.requires_grad] + \
                   [p for p in eff_features[-3:].parameters() if p.requires_grad]
head_params = (
    list(joint_model.resnet_projection.parameters()) +
    list(joint_model.efficientnet_projection.parameters()) +
    list(joint_model.attention.parameters()) +
    list(joint_model.classifier.parameters())
)

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": 1e-5, "weight_decay": 1e-4},
    {"params": head_params, "lr": 1e-4, "weight_decay": 1e-4}
])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
n_trainable = sum(p.numel() for p in joint_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in joint_model.parameters())
print(f"✓ Joint Fine-Tuned Model ready. Trainable params: {n_trainable:,} / {n_total:,}")


## Dataset, Augmentations & DataLoaders

In [ ]:
class RetinalFundusDataset(Dataset):
    def __init__(self, df, img_root, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_root = Path(img_root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p_path = str(row["processed_path"])
        if os.path.exists(p_path):
            full_path = p_path
        else:
            filename = os.path.basename(p_path)
            dataset_name = row["dataset"]
            full_path = os.path.join(self.img_root, dataset_name, filename)
            if not os.path.exists(full_path):
                full_path = os.path.join(self.img_root, filename)
        try:
            img = Image.open(full_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), color=(0, 0, 0))
        label = int(row["label_dr_standard"]) if not pd.isna(row["label_dr_standard"]) else 0
        tensor_img = self.transform(img) if self.transform else transforms.ToTensor()(img)
        return tensor_img, label


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Light augmentation for fine-tuning — fundus images are orientation-agnostic
# (flips are safe) but should NOT be rotated by arbitrary large angles or have
# color drastically shifted, since lesion color/contrast is diagnostic.
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = RetinalFundusDataset(train_df, IMG_ROOT, transform=train_transforms)
val_dataset = RetinalFundusDataset(val_df, IMG_ROOT, transform=eval_transforms)
test_dataset = RetinalFundusDataset(test_df, IMG_ROOT, transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


## Training Loop: AMP, Gradient Accumulation, Differential LR, Checkpointing

Uses `torch.cuda.amp.autocast` + `GradScaler` for mixed-precision training on the Kaggle T4/P100 GPUs. Effective batch size stays 32 via `ACCUM_STEPS` gradient accumulation (set `ACCUM_STEPS > 1` to fine-tune with a physical batch size of 16 if VRAM is tight). Best checkpoint is selected by **validation QWK** (the clinically relevant ordinal metric), not validation loss.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_n = 0.0, 0
    all_true, all_pred, all_prob = [], [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                             dtype=torch.float16, enabled=(device.type == "cuda")):
            logits, _ = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        total_n += images.size(0)
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()
        all_true.extend(labels.cpu().numpy().tolist())
        all_pred.extend(probs.argmax(axis=1).tolist())
        all_prob.append(probs)
    y_true = np.array(all_true)
    y_pred = np.array(all_pred)
    y_prob = np.concatenate(all_prob, axis=0)
    acc = accuracy_score(y_true, y_pred)
    _, _, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    return {
        "loss": total_loss / max(total_n, 1), "accuracy": acc, "macro_f1": f1_macro,
        "qwk": qwk, "y_true": y_true, "y_pred": y_pred, "y_prob": y_prob
    }


def train_one_epoch(model, loader, criterion, optimizer, device, scaler, accum_steps=1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad(set_to_none=True)
    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda" if device.type == "cuda" else "cpu",
                             dtype=torch.float16, enabled=(device.type == "cuda")):
            logits, _ = model(images)
            loss = criterion(logits, labels) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * accum_steps * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
    return running_loss / max(total, 1), correct / max(total, 1)


In [ ]:
EPOCHS = 10
ACCUM_STEPS = 1  # bump to 2 with a smaller physical batch size if VRAM is constrained

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
BEST_CKPT_PATH = os.path.join(CKPT_OUT_DIR, "best_joint_finetuned_attention_fusion.pth")

history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": [], "val_qwk": [], "val_macro_f1": []}
best_val_qwk = -float("inf")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_accuracy = train_one_epoch(
        joint_model, train_loader, criterion, optimizer, device, scaler, accum_steps=ACCUM_STEPS
    )
    val_metrics = evaluate(joint_model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_metrics["loss"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    history["val_qwk"].append(val_metrics["qwk"])
    history["val_macro_f1"].append(val_metrics["macro_f1"])

    scheduler.step(val_metrics["loss"])

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.4f} | "
          f"Val Macro F1: {val_metrics['macro_f1']:.4f} | Val QWK: {val_metrics['qwk']:.4f}")

    if val_metrics["qwk"] > best_val_qwk:
        best_val_qwk = val_metrics["qwk"]
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": joint_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_qwk": val_metrics["qwk"],
        }, BEST_CKPT_PATH)
        print(f"✓ New best checkpoint saved (Val QWK: {best_val_qwk:.4f}) -> {BEST_CKPT_PATH}")

print(f"\n✓ Training complete. Best Val QWK: {best_val_qwk:.4f}")


## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history["train_accuracy"], label="Train Acc")
axes[1].plot(history["val_accuracy"], label="Val Acc")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history["val_qwk"], label="Val QWK", color="crimson")
axes[2].plot(history["val_macro_f1"], label="Val Macro F1", color="darkorange")
axes[2].set_title("Ordinal / Macro Metrics"); axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/joint_finetuning_curves.png", dpi=300, bbox_inches="tight")
plt.show()


## Final Test-Set Evaluation

Loads the best checkpoint (by validation QWK) and reports test performance against the single-model baseline (Macro F1 > 0.7087, QWK > 0.7628, Acc > 0.7431).

In [ ]:
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
joint_model.load_state_dict(best_ckpt["model_state_dict"])
joint_model.eval()

test_metrics = evaluate(joint_model, test_loader, criterion, device)
test_balanced_acc = balanced_accuracy_score(test_metrics["y_true"], test_metrics["y_pred"])
test_mcc = matthews_corrcoef(test_metrics["y_true"], test_metrics["y_pred"])

print("=== Joint Fine-Tuned Attention Fusion — Test Set Performance ===")
print(f"Accuracy:          {test_metrics['accuracy']:.4f}")
print(f"Macro F1:          {test_metrics['macro_f1']:.4f}")
print(f"Balanced Accuracy: {test_balanced_acc:.4f}")
print(f"QWK:               {test_metrics['qwk']:.4f}")
print(f"MCC:               {test_mcc:.4f}")

print("\n=== Comparison vs Single-Model Baseline (Weighted ResNet-18) ===")
print(f"Macro F1: {test_metrics['macro_f1']:.4f} vs 0.7087 baseline "
      f"({'BETTER' if test_metrics['macro_f1'] > 0.7087 else 'WORSE/EQUAL'})")
print(f"QWK:      {test_metrics['qwk']:.4f} vs 0.7628 baseline "
      f"({'BETTER' if test_metrics['qwk'] > 0.7628 else 'WORSE/EQUAL'})")
print(f"Accuracy: {test_metrics['accuracy']:.4f} vs 0.7431 baseline "
      f"({'BETTER' if test_metrics['accuracy'] > 0.7431 else 'WORSE/EQUAL'})")

results_summary = pd.DataFrame([{
    "Model": "Joint Fine-Tuned Attention Fusion",
    "Accuracy": round(float(test_metrics["accuracy"]), 4),
    "Macro F1": round(float(test_metrics["macro_f1"]), 4),
    "Balanced Accuracy": round(float(test_balanced_acc), 4),
    "QWK": round(float(test_metrics["qwk"]), 4),
    "MCC": round(float(test_mcc), 4),
}])
results_summary.to_csv(f"{TABLE_DIR}/joint_finetuning_test_results.csv", index=False)
display(results_summary)
